
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 1 - Exploring Spark Architecture in Databricks

In this demonstration, we'll explore how Spark's architecture components manifest in a Databricks environment and how to monitor them using the various UIs available to us.

### Objectives
- Identify key Spark architecture components in a Databricks cluster
- Navigate the Spark UI to monitor application execution
- Understand how Databricks implements Spark's cluster management

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A: Introduction to Notebooks

We will run all of the demos and exercises for this course in notebooks. Notebooks are interactive documents that combine live code, visualizations, narrative text, and outputs in a single document. In Databricks, notebooks provide a powerful environment for data exploration, analysis, and collaboration.

Key features of Databricks notebooks:
- Code execution: Run code in multiple languages (Python, SQL, R, Scala)
- Rich text formatting: Support for Markdown to create well-documented analyses
- Cell-based structure: Code and content is organized into executable cells
- Interactive visualizations: Direct plotting and charting of results
- Collaboration: Share and work together on notebooks in real-time

The default language for a notebook is set when you create it, as indicated by the __Python__ selection in the toolbar above☝️. You can change the default language through the notebook settings. Additionally, you can use magic commands to execute code in different languages within the same notebook:

- Use `%python` to execute Python code
- Use `%sql` to execute SQL queries
- Use `%r` to execute R code
- Use `%scala` to execute Scala code

The `%run` magic command is particularly useful - it allows you to execute another notebook within your current notebook, enabling modular code organization and reuse. For example `%run /path/to/another/notebook`.  

This will execute all the cells in the referenced notebook as if they were part of your current notebook.

## B. The SparkSession and SparkContext

The SparkSession is automatically instantiated as `spark` in Databricks notebooks connected to a cluster.  The SparkContext is available via the SparkSession using `spark.sparkContext` or simply `sc`.

In [0]:
spark

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
application_id = spark.sparkContext.applicationId
print(f"Application ID: {application_id}")

Application ID: local-1751001184666


In [0]:
from pyspark.sql import SparkSession

# Create the first SparkSession
spark1 = SparkSession.builder.appName("FirstSession").getOrCreate()

# Create a new SparkSession using newSession()
spark2 = spark1.newSession()

# Both sessions will have the same SparkContext
print(spark1.sparkContext == spark2.sparkContext) # This will print: True

True


In [0]:
%scala
spark

res0: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@525c8a9a

In [0]:
sc

<SparkContext master=local[*, 4] appName=Databricks Shell>

## C. Understanding Your Databricks Cluster

1. Right click on the __Compute__  menu item, select __Open link in new tab__

2. Locate your cluster in the __All-purpose compute__ tab, here you can see high level information about the cluster including memory and cores 

3. Click on the link to the cluster under the __Name__ column, here you can see more detailed information and access logs and metrics
<!-- 4. Click on the "Spark UI" button under Compute to observe:
    - The DAG visualization showing stages
    - Executor information showing Worker nodes
    - Memory usage across your cluster -->

## D. Creating and Monitoring a Spark Job

Let's create a simple job that will help us visualize the execution flow. Run the below cell to create a spark Job

**NOTE:** Don't focus too much on the code here. We want to focus on exploring the access logs and metrics.

In [0]:
# Create a large DataFrame to see parallelization in action
from pyspark.sql.functions import *
import time

# Generate some data
df = spark.range(0, 1000000)
df = df.withColumn("square", col("id") * col("id"))

# # Force multiple stages with a shuffle operation
result = df.groupBy(col("id") % 100).agg(sum("square").alias("sum_squares"))

# result = df.repartition(2)
# Cache the result to see storage in the UI
result.cache()

# Force the computation with the 'count' action
# print(f"Number of groups: {result.count()}")
result.collect()

[Row((id % 100)=26, sum_squares=3333093330760000),
 Row((id % 100)=29, sum_squares=3333123329410000),
 Row((id % 100)=65, sum_squares=3333483327250000),
 Row((id % 100)=19, sum_squares=3333023334610000),
 Row((id % 100)=54, sum_squares=3333373325160000),
 Row((id % 100)=0, sum_squares=3332833350000000),
 Row((id % 100)=22, sum_squares=3333053332840000),
 Row((id % 100)=7, sum_squares=3332903343490000),
 Row((id % 100)=77, sum_squares=3333603332290000),
 Row((id % 100)=34, sum_squares=3333173327560000),
 Row((id % 100)=50, sum_squares=3333333325000000),
 Row((id % 100)=94, sum_squares=3333773344360000),
 Row((id % 100)=57, sum_squares=3333403325490000),
 Row((id % 100)=32, sum_squares=3333153328240000),
 Row((id % 100)=43, sum_squares=3333263325490000),
 Row((id % 100)=84, sum_squares=3333673336560000),
 Row((id % 100)=31, sum_squares=3333143328610000),
 Row((id % 100)=39, sum_squares=3333223326210000),
 Row((id % 100)=98, sum_squares=3333813348040000),
 Row((id % 100)=25, sum_squares=3

In [0]:
# Force the computation with the 'count' action
print(f"Number of groups: {result.count()}")

Number of groups: 100


### Exploring the Spark UI

> NOTE: Right click on the __View__ link (if there are more than one link, click the last one) under the __Spark Jobs__ heading ☝️ and open in a new tab

Now that we have an active job, let's explore key areas of the Spark UI:

1. **Jobs Tab**
   - Shows the DAG for our groupBy operation
   - Multiple stages due to the shuffle operation
   - Click the link inside the description tab of a stage to see task-level details.


2. **Executors Tab**
   - Lists all executors and their resource usage
   - Shows how many cores and memory each executor has
   - Demonstrates the Worker node distribution

3. **Storage Tab**
   - Shows cached DataFrames
   - Displays memory usage across executors

Notice how the architecture we discussed (Driver → Master → Workers → Executors) is reflected in the UI. The Driver coordinates the job, while Executors on Worker nodes perform the actual computations (although in this class all processes reside on a single node).

In [0]:
# Free up executor memory by unpersisting cached objects
result.unpersist()

DataFrame[(id % 100): bigint, sum_squares: bigint]


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
